# Notebook de forecasting

Este notebook carga los datos de inferencia y prepara el entorno para realizar forecasting sobre las ventas de 2025.

## 1. Importar librerías necesarias

Importamos las mismas librerías utilizadas en el notebook de entrenamiento.

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
import holidays
import streamlit as st

## 2. Cargar archivo de inferencia en un DataFrame

Cargamos el archivo `ventas_2025_inferencia.csv` ubicado en `data/raw/inferencia` en un DataFrame llamado `inferencia_df`.

In [2]:
# Cargar archivo de inferencia en un DataFrame
inferencia_path = "../data/raw/inferencia/ventas_2025_inferencia.csv"
inferencia_df = pd.read_csv(inferencia_path, encoding="utf-8", sep=",")

# Conversión de fecha a datetime si existe la columna 'fecha'
if 'fecha' in inferencia_df.columns:
    inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'], errors='coerce')

print("Inferencia:", inferencia_df.shape)
display(inferencia_df.head())

Inferencia: (888, 13)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


## 3. Transformaciones y feature engineering sobre inferencia_df

Aplicamos exactamente las mismas transformaciones que se realizaron sobre `df` en el notebook de entrenamiento:
- Descuento porcentual
- Variables temporales y de calendario (festivos España, Black Friday, Cyber Monday…)
- Variables lag (1-7 días) y media móvil de 7 días de `unidades_vendidas`
- Precio medio de la competencia y ratio de precio
- Copia y one-hot encoding de `nombre`, `categoria`, `subcategoria`

In [3]:
# Paso 1: Descuento porcentual (igual que en entrenamiento)
inferencia_df['descuento_porcentaje'] = (
    (inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base']
) * 100

print("descuento_porcentaje creado.")
inferencia_df[['fecha', 'producto_id', 'precio_base', 'precio_venta', 'descuento_porcentaje']].head()

descuento_porcentaje creado.


,fecha,producto_id,precio_base,precio_venta,descuento_porcentaje
0,2025-10-25,PROD_001,115,113.13,-1.626087
1,2025-10-25,PROD_002,135,141.89,5.103704
2,2025-10-25,PROD_003,85,85.79,0.929412
3,2025-10-25,PROD_004,75,76.19,1.586667
4,2025-10-25,PROD_005,35,35.48,1.371429


In [4]:
# Paso 2: Variables temporales y de calendario (igual que en entrenamiento)
es_holidays_inf = holidays.country_holidays('ES', years=inferencia_df['fecha'].dt.year.unique())

inferencia_df['anio'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.weekday
inferencia_df['nombre_dia'] = inferencia_df['fecha'].dt.day_name(locale='es_ES')
inferencia_df['es_fin_de_semana'] = inferencia_df['dia_semana'].isin([5, 6]).astype(int)
inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(es_holidays_inf).astype(int)

def es_black_friday(fecha):
    if fecha.month == 11:
        ultimo_viernes = max([d for d in pd.date_range(start=fecha.replace(day=1), end=fecha.replace(day=30)) if d.weekday() == 4])
        return int(fecha == ultimo_viernes)
    return 0

def es_cyber_monday(fecha):
    if fecha.month == 11 or fecha.month == 12:
        dias_nov = pd.date_range(start=fecha.replace(month=11, day=1), end=fecha.replace(month=11, day=30))
        black_friday = max([d for d in dias_nov if d.weekday() == 4])
        cyber_monday = black_friday + pd.Timedelta(days=3)
        return int(fecha == cyber_monday)
    return 0

inferencia_df['es_black_friday'] = inferencia_df['fecha'].apply(es_black_friday)
inferencia_df['es_cyber_monday'] = inferencia_df['fecha'].apply(es_cyber_monday)
inferencia_df['dia_anio'] = inferencia_df['fecha'].dt.dayofyear
inferencia_df['semana_anio'] = inferencia_df['fecha'].dt.isocalendar().week
inferencia_df['es_primer_dia_mes'] = (inferencia_df['dia_mes'] == 1).astype(int)
inferencia_df['es_ultimo_dia_mes'] = (inferencia_df['fecha'] == inferencia_df['fecha'] + pd.offsets.MonthEnd(0)).astype(int)
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['es_festivo_o_finde'] = ((inferencia_df['es_festivo'] == 1) | (inferencia_df['es_fin_de_semana'] == 1)).astype(int)
inferencia_df['es_navidad'] = (inferencia_df['fecha'].dt.month.eq(12) & inferencia_df['fecha'].dt.day.isin([24])).astype(int)
inferencia_df['es_fin_anio'] = (
    (inferencia_df['fecha'].dt.month.eq(12) & inferencia_df['fecha'].dt.day.eq(31)) |
    (inferencia_df['fecha'].dt.month.eq(1) & inferencia_df['fecha'].dt.day.eq(1))
).astype(int)

print("Variables temporales y de calendario creadas.")
inferencia_df.head()

C:\Users\usuario\AppData\Local\Temp\ipykernel_5432\845643746.py:10: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(es_holidays_inf).astype(int)


Variables temporales y de calendario creadas.


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,es_black_friday,es_cyber_monday,dia_anio,semana_anio,es_primer_dia_mes,es_ultimo_dia_mes,trimestre,es_festivo_o_finde,es_navidad,es_fin_anio
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,...,0,0,298,43,0,0,4,1,0,0
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,...,0,0,298,43,0,0,4,1,0,0
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,...,0,0,298,43,0,0,4,1,0,0
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,...,0,0,298,43,0,0,4,1,0,0
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,...,0,0,298,43,0,0,4,1,0,0


In [5]:
# Paso 3: Variables lag (1-7 días) y media móvil de 7 días de unidades_vendidas
# Se calcula por producto y año para mantener la misma lógica que en entrenamiento.
# Los datos de octubre quedan en el df para que los lags de noviembre sean válidos.

def crear_lags_y_media_df(df):
    df = df.sort_values(['producto_id', 'anio', 'fecha'])
    for lag in range(1, 8):
        df[f'unidades_vendidas_lag{lag}'] = df.groupby(['producto_id', 'anio'])['unidades_vendidas'].shift(lag)
    df['unidades_vendidas_mm7'] = df.groupby(['producto_id', 'anio'])['unidades_vendidas'].transform(
        lambda x: x.rolling(window=7, min_periods=1).mean().shift(1)
    )
    return df

inferencia_df = crear_lags_y_media_df(inferencia_df)
cols_lag_mm = [f'unidades_vendidas_lag{i}' for i in range(1, 8)] + ['unidades_vendidas_mm7']

print("Lags y media móvil creados.")
inferencia_df[['fecha', 'producto_id', 'unidades_vendidas'] + cols_lag_mm].head(10)

Lags y media móvil creados.


,fecha,producto_id,unidades_vendidas,unidades_vendidas_lag1,unidades_vendidas_lag2,unidades_vendidas_lag3,unidades_vendidas_lag4,unidades_vendidas_lag5,unidades_vendidas_lag6,unidades_vendidas_lag7,unidades_vendidas_mm7
0,2025-10-25,PROD_001,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,2025-10-26,PROD_001,20.0,26.0,NaN,NaN,NaN,NaN,NaN,NaN,26.000000
48,2025-10-27,PROD_001,16.0,20.0,26.0,NaN,NaN,NaN,NaN,NaN,23.000000
72,2025-10-28,PROD_001,15.0,16.0,20.0,26.0,NaN,NaN,NaN,NaN,20.666667
96,2025-10-29,PROD_001,14.0,15.0,16.0,20.0,26.0,NaN,NaN,NaN,19.250000
120,2025-10-30,PROD_001,10.0,14.0,15.0,16.0,20.0,26.0,NaN,NaN,18.200000
144,2025-10-31,PROD_001,14.0,10.0,14.0,15.0,16.0,20.0,26.0,NaN,16.833333
168,2025-11-01,PROD_001,NaN,14.0,10.0,14.0,15.0,16.0,20.0,26.0,16.428571
192,2025-11-02,PROD_001,NaN,NaN,14.0,10.0,14.0,15.0,16.0,20.0,14.833333
216,2025-11-03,PROD_001,NaN,NaN,NaN,14.0,10.0,14.0,15.0,16.0,13.800000


In [6]:
# Paso 4: Precio medio de la competencia y ratio de precio (igual que en entrenamiento)
inferencia_df['precio_competencia'] = inferencia_df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)
inferencia_df['ratio_precio_competencia'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']
inferencia_df = inferencia_df.drop(['Amazon', 'Decathlon', 'Deporvillage'], axis=1)

print("precio_competencia y ratio_precio_competencia creados.")
inferencia_df[['fecha', 'producto_id', 'precio_venta', 'precio_competencia', 'ratio_precio_competencia']].head()

precio_competencia y ratio_precio_competencia creados.


,fecha,producto_id,precio_venta,precio_competencia,ratio_precio_competencia
0,2025-10-25,PROD_001,113.13,102.573333,1.102918
24,2025-10-26,PROD_001,105.75,98.356667,1.075169
48,2025-10-27,PROD_001,114.95,97.740000,1.176079
72,2025-10-28,PROD_001,117.31,103.146667,1.137313
96,2025-10-29,PROD_001,108.10,100.520000,1.075408


In [7]:
# Paso 5: Copia de variables y one-hot encoding (igual que en entrenamiento)
for col in ['nombre', 'categoria', 'subcategoria']:
    inferencia_df[f'{col}_h'] = inferencia_df[col]

inferencia_df = pd.get_dummies(inferencia_df, columns=['nombre_h', 'categoria_h', 'subcategoria_h'], drop_first=False)

# Alinear columnas OHE con las del DataFrame de entrenamiento (df.csv)
# para garantizar que inferencia_df tenga exactamente las mismas variables
df_train_cols = pd.read_csv('../data/processed/df.csv', nrows=0).columns.tolist()
ohe_cols_train = [c for c in df_train_cols if c.startswith(('nombre_h_', 'categoria_h_', 'subcategoria_h_'))]
for c in ohe_cols_train:
    if c not in inferencia_df.columns:
        inferencia_df[c] = False

print(f"One-hot encoding aplicado. Shape actual: {inferencia_df.shape}")
inferencia_df.head(2)

One-hot encoding aplicado. Shape actual: (888, 82)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,...,False,False,False,False,False,False,False,False,True,False
24,2025-10-26,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,20.0,105.75,2115.00,...,False,False,False,False,False,False,False,False,True,False


## 4. Filtrado de noviembre y guardado

Eliminamos los registros de octubre (usados solo para calcular los lags) y guardamos únicamente los registros de noviembre 2025 en `data/processed/inferencia_df_transformado.csv`.

In [ ]:
# Eliminar registros de octubre (solo se usaron para calcular los lags)
# y mantener únicamente los registros de noviembre 2025
inferencia_df = inferencia_df[inferencia_df['fecha'].dt.month == 11].reset_index(drop=True)

# Guardar el DataFrame transformado en data/processed
ruta_guardado = 'data/processed/inferencia_df_transformado.csv'
inferencia_df.to_csv(ruta_guardado, index=False, encoding='utf-8-sig', errors='replace')

print(f"DataFrame de inferencia guardado en: {ruta_guardado}")
print(f"Shape final (solo noviembre): {inferencia_df.shape}")
display(inferencia_df.head())

DataFrame de inferencia guardado en: ../data/processed/inferencia_df_transformado.csv
Shape final (solo noviembre): (720, 82)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
0,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
1,2025-11-02,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
2,2025-11-03,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
3,2025-11-04,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
4,2025-11-05,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False


In [24]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'descuento_porcentaje', 'anio', 'mes', 'dia_mes',
       'dia_semana', 'nombre_dia', 'es_fin_de_semana', 'es_festivo',
       'es_black_friday', 'es_cyber_monday', 'dia_anio', 'semana_anio',
       'es_primer_dia_mes', 'es_ultimo_dia_mes', 'trimestre',
       'es_festivo_o_finde', 'es_navidad', 'es_fin_anio',
       'unidades_vendidas_lag1', 'unidades_vendidas_lag2',
       'unidades_vendidas_lag3', 'unidades_vendidas_lag4',
       'unidades_vendidas_lag5', 'unidades_vendidas_lag6',
       'unidades_vendidas_lag7', 'unidades_vendidas_mm7', 'precio_competencia',
       'ratio_precio_competencia', 'nombre_h_Adidas Own The Run Jacket',
       'nombre_h_Adidas Ultraboost 23', 'nombre_h_Asics Gel Nimbus 25',
       'nombre_h_Bowflex SelectTech 552', 'nombre_h_Columbia Silver Ridge',
       'nombre_h_Decathlon Bandas Elásticas Set'